In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb

print("==========================================================")
print("  PHASE 1: CHRONOLOGICAL TIMELINES & TIME-DECAY MATRIX    ")
print("==========================================================")

max_time = df['timestamp'].max()
test_split = max_time - pd.Timedelta(days=5)       # Final 5 days for testing
lgb_split = test_split - pd.Timedelta(days=5)      # Middle 5 days for LightGBM features

# Chronological split to prevent future-looking leakage
logs_retrieval = df[df['timestamp'] < lgb_split].copy()
logs_train_lgb = df[(df['timestamp'] >= lgb_split) & (df['timestamp'] < test_split)].copy()
logs_test_box = df[df['timestamp'] >= test_split].copy()

# 1. Compute global popularity background signal
global_pop = logs_retrieval.groupby('show_id').size().reset_index(name='global_views')
global_pop['show_global_popularity_log'] = np.log1p(global_pop['global_views'])
top_500_candidates = global_pop.sort_values(by='show_global_popularity_log', ascending=False)['show_id'].head(500).values

print("Compiling real-time user session chronological timelines...")
# Sort historical logs to compute sequence tracking arrays
logs_retrieval = logs_retrieval.sort_values(by=['user_id', 'timestamp'])

# Build user-centric real-time state dictionaries for ultra-fast lookup
user_last_three_shows = logs_retrieval.groupby('user_id')['show_id'].apply(lambda x: list(x[-3:])).to_dict()
user_most_frequent_show = logs_retrieval.groupby('user_id')['show_id'].apply(lambda x: x.mode().iloc[0] if not x.empty else 0).to_dict()

# Calculate time decay matrix: weight interactions based on how recently they happened
# Halflife of 4 days: interactions from 4 days ago are worth half as much
logs_retrieval['days_ago'] = (lgb_split - logs_retrieval['timestamp']).dt.total_seconds() / (24 * 3600)
logs_retrieval['time_decay_weight'] = 0.5 ** (logs_retrieval['days_ago'] / 4.0)

# Build personalized sequential recency profile
user_sequential_affinity = logs_retrieval.groupby(['user_id', 'show_id'])['time_decay_weight'].sum().to_dict()

# ==============================================================================
# PHASE 2: SEQUENTIAL FEATURE ENGINEERING GRID
# ==============================================================================
print("Generating sequential feature matrices across candidate timelines...")
unique_train_users = logs_train_lgb['user_id'].unique()
sequential_rows = []

for user_id in unique_train_users:
    user_timeline = logs_train_lgb[logs_train_lgb['user_id'] == user_id]
    seen_in_window = set(user_timeline['show_id'])
    
    # Extract structural sequence states
    last_3 = user_last_three_shows.get(user_id, [])
    favorite_show = user_most_frequent_show.get(user_id, -1)
    
    # Evaluate candidates inside the accessible global pool
    for show_id in top_500_candidates:
        is_positive = 1 if show_id in seen_in_window else 0
        
        # Feature 1: Explicit time-decay score
        decay_affinity = user_sequential_affinity.get((user_id, show_id), 0.0)
        
        # Feature 2: Immediate Short-Term Session Momentum (Recency Match)
        immediate_recency_rank = 0
        if show_id in last_3:
            immediate_recency_rank = 3 - last_3.index(show_id) # 3 if most recent, 1 if 3rd back
            
        # Feature 3: Serial Long-Term Anchor Match
        is_favorite_anchor = 1 if show_id == favorite_show else 0
        
        sequential_rows.append({
            'user_id': user_id,
            'show_id': show_id,
            'relevance': is_positive,
            'seq_decay_affinity_score': decay_affinity,
            'seq_immediate_recency_rank': immediate_recency_rank,
            'seq_is_favorite_anchor': is_favorite_anchor
        })

ranking_train_df = pd.DataFrame(sequential_rows)
ranking_train_df = ranking_train_df.merge(global_pop, on='show_id', how='left').fillna(0)
ranking_train_df = ranking_train_df.merge(show_features, on='show_id', how='left', suffixes=('', '_meta')).fillna(0)
ranking_train_df = ranking_train_df.merge(user_features, on='user_id', how='left').fillna(0)

# Compound Multipliers
ranking_train_df['seq_pop_interaction_leverage'] = ranking_train_df['seq_decay_affinity_score'] * ranking_train_df['show_global_popularity_log']

sequential_features = [
    'show_global_popularity_log',
    'show_total_watch_time_log',
    'user_total_interactions',
    'user_avg_interaction_score',
    'seq_decay_affinity_score',       # The time-decay sequence vector
    'seq_immediate_recency_rank',     # Short-term session momentum
    'seq_is_favorite_anchor',         # Historical anchor show
    'seq_pop_interaction_leverage'    # Interaction cross
]

# Ensure grouping constraints align for LambdaRank execution
ranking_train_df = ranking_train_df.sort_values(by='user_id').reset_index(drop=True)
train_group_counts = ranking_train_df.groupby('user_id').size().values

# ==============================================================================
# PHASE 3: TRAINING THE SEQUENTIAL LAMBDARANK TREE
# ==============================================================================
X_train = ranking_train_df[sequential_features]
y_train = ranking_train_df['relevance']
train_dataset = lgb.Dataset(X_train, label=y_train, group=train_group_counts)

seq_params = {
    'objective': 'lambdarank', 'metric': 'map', 'map_eval_at': [10],
    'learning_rate': 0.05, 'max_depth': 8, 'num_leaves': 63, 'verbosity': -1, 'seed': 42
}
print("Training Tree Ranker with Sequence-Aware Temporal Tracking features...")
sequential_ranker_model = lgb.train(params=seq_params, train_set=train_dataset, num_boost_round=150)

# ==============================================================================
# PHASE 4: TIME-AWARE PRODUCTION EVALUATION ENGINE
# ==============================================================================
class PhiloSequentialEvaluationEngine:
    @staticmethod
    def evaluate_pipeline(lgb_ranker, test_logs, show_features, user_features, feature_cols, 
                          top_candidates, user_decay_profile, last_three_dict, favorite_dict, top_n=10):
        print(f"Beginning Real-Time Sequence Evaluation Loop...")
        
        test_ground_truth = test_logs.groupby('user_id')['show_id'].apply(set).reset_index()
        test_ground_truth.columns = ['user_id', 'actual_shows_watched']
        
        show_feat_dict = show_features.set_index('show_id')[['show_global_popularity_log', 'show_total_watch_time_log']].to_dict('index')
        user_feat_dict = user_features.set_index('user_id')[['user_total_interactions', 'user_avg_interaction_score']].to_dict('index')
        
        avg_user_interactions = user_features['user_total_interactions'].mean()
        avg_user_score = user_features['user_avg_interaction_score'].mean()
        
        recalls = []
        mrrs = []
        
        test_sample = test_ground_truth.sample(n=min(3000, len(test_ground_truth)), random_state=42)
        
        for idx, row in test_sample.iterrows():
            user_id = row['user_id']
            actual_shows = row['actual_shows_watched']
            
            u_feats = user_feat_dict.get(user_id, {'user_total_interactions': avg_user_interactions, 'user_avg_interaction_score': avg_user_score})
            last_3 = last_three_dict.get(user_id, [])
            fav_anchor = favorite_dict.get(user_id, -1)
            
            candidate_rows = []
            for show_id in top_candidates:
                s_feats = show_feat_dict.get(show_id, {'show_global_popularity_log': 0, 'show_total_watch_time_log': 0})
                
                decay_affinity = user_decay_profile.get((user_id, show_id), 0.0)
                immediate_recency = 3 - last_3.index(show_id) if show_id in last_3 else 0
                is_fav = 1 if show_id == fav_anchor else 0
                pop_leverage = decay_affinity * s_feats['show_global_popularity_log']
                
                candidate_rows.append([
                    s_feats['show_global_popularity_log'],
                    s_feats['show_total_watch_time_log'],
                    u_feats['user_total_interactions'],
                    u_feats['user_avg_interaction_score'],
                    decay_affinity,
                    immediate_recency,
                    is_fav,
                    pop_leverage
                ])
                
            X_candidates = pd.DataFrame(candidate_rows, columns=feature_cols)
            predicted_scores = lgb_ranker.predict(X_candidates)
            
            top_indices = np.argsort(predicted_scores)[::-1][:top_n]
            final_predictions = [top_candidates[i] for i in top_indices]
            
            hits = [show for show in final_predictions if show in actual_shows]
            recall_score = len(hits) / min(len(actual_shows), top_n) if len(actual_shows) > 0 else 0
            recalls.append(recall_score)
            
            mrr_score = 0
            for rank_idx, show in enumerate(final_predictions, 1):
                if show in actual_shows:
                    mrr_score = 1.0 / rank_idx
                    break
            mrrs.append(mrr_score)
            
        print("\n==========================================================")
        print("          TEMPORAL SEQUENTIAL PIPELINE METRICS            ")
        print("==========================================================")
        print(f"  - Final Sequence Recall@10             : {np.mean(recalls)*100:.2f}%")
        print(f"  - Final Sequence MRR@10                : {np.mean(mrrs):.4f}")
        print("==========================================================\n")

# Execute final time-aware production run
PhiloSequentialEvaluationEngine.evaluate_pipeline(
    lgb_ranker=sequential_ranker_model, test_logs=logs_test_box,
    show_features=show_features, user_features=user_features,
    feature_cols=sequential_features, top_candidates=top_500_candidates,
    user_decay_profile=user_sequential_affinity, last_three_dict=user_last_three_shows,
    favorite_dict=user_most_frequent_show
)

  PHASE 1: CHRONOLOGICAL TIMELINES & TIME-DECAY MATRIX    


NameError: name 'df' is not defined